# 02 Knowledge Base and RAG

**Owner.** Jack (Product Lead & Knowledge Architect) · **Supports.** Tim

**Goal.** Turn the articles of the Swiss Code of Obligations into a searchable knowledge base, so that the agent
answers from the real text instead of from memory.

**Definition of Done.** Index built from the official Fedlex text for OR 253 ff. and 319 ff. Hit@k measured on the
test cases. Chunking decision written into the decision log (tasks T11 and T12).

**Prerequisite.** PR #24 (T11) merged and the GitHub integration `swiss_legal_agent` pulled to the latest `main`.
Run this notebook before 03 and 04. It uses `data/or_articles.jsonl` from the repository.

## 1. Setup

In [ ]:
import os, sys, json, re, pathlib, subprocess
from dotenv import load_dotenv

load_dotenv()
LLM_BASE_URL = "https://litellm.engines.aisl.science/v1"
EMBED_MODEL = "andreasmartin/apertus-v1.1-swiss-embed-0.4b"   # Swiss embedding model on the FHNW proxy
EMBEDDING_BACKEND = "proxy"     # "proxy" or "local" (sentence-transformers, no key needed)
USE_RERANKER = False            # cross-encoder reranking, section 6

# The GitHub integration mounts the repository next to the notebooks (folder swiss_legal_agent, branch main).
REPO_ROOT = pathlib.Path("swiss_legal_agent").resolve()
sys.path.insert(0, str(REPO_ROOT))
from lexgraph.rag.ingest import ARTICLES_FILE, FEDLEX_VERSION, load_articles, to_documents

DATA_DIR = ARTICLES_FILE.parent   # swiss_legal_agent/data

## 2. Source data

The OR is SR 220 on Fedlex. We keep **one record per article** with the fields the citation check needs later:

`{"id": "OR-335c", "law": "OR", "article": "335c", "area": "arbeitsrecht" | "mietrecht", "title": ..., "context": ..., "text": ..., "url": ..., "version": "2026-10-01"}`

**Ingestion (T11).** `python -m lexgraph.rag.ingest` downloads the Akoma Ntoso XML of the consolidated German OR
from the Fedlex file store, pinned to the version of 1 Oct 2026 (`FEDLEX_VERSION`), and parses it with `parse_fedlex`:
Art. 253 to 273c (tenancy, 79 articles) and 319 to 362 (employment, 137 articles), 216 in total. Repealed articles
(274 to 274g, 332a, 343) and footnotes are left out. `title` is the nearest margin heading, `context` the whole
heading path, `url` the real Fedlex anchor (`#art_335_c`).

The result `data/or_articles.jsonl` is versioned in the repository, so the cell below only reads it. The six
paraphrased `SAMPLE` records stay in `lexgraph/rag/ingest.py` for the offline tests and are **not** part of the
knowledge base. Federal Supreme Court decisions (BGE) would follow as a second file with the same structure (optional).

In [ ]:
# T11: the knowledge base is data/or_articles.jsonl in the repository (official Fedlex text, pinned version).
# It is only rebuilt if the file is missing, or with REBUILD = True after FEDLEX_VERSION was changed.
REBUILD = False
if REBUILD or not ARTICLES_FILE.exists():
    subprocess.run([sys.executable, "-m", "lexgraph.rag.ingest"], cwd=REPO_ROOT, check=True)

articles = load_articles()
print(len(articles), "articles, Fedlex version", FEDLEX_VERSION)

# Check: both areas complete, no paraphrased sample records, real Fedlex anchor
by_area = {a: sum(r["area"] == a for r in articles) for a in ("mietrecht", "arbeitsrecht")}
print(by_area)   # expected: {'mietrecht': 79, 'arbeitsrecht': 137}
assert not any(r.get("placeholder") for r in articles), "sample records in the knowledge base"
example = next(r for r in articles if r["id"] == "OR-335c")
print(example["context"])
print(example["url"])   # ends with #art_335_c
print(example["text"][:300])

## 3. Chunking: one chunk per article

An article is the unit a lawyer cites. If a chunk is exactly one article, every hit already carries its citation,
and the citation check in notebook 05 can compare article numbers directly.

The usual alternative, fixed chunks of about 500 tokens, cuts articles in the middle and mixes neighbouring
articles in one chunk. On the real text one article per chunk works well: 208 of 216 articles are under 1500
characters, the longest (Art. 362, 3642 characters) is a list of articles. Very long articles can still be split per
paragraph in T12, keeping the article id.

The heading path (`context`) is part of the indexed text. It adds the words of the margin headings, for example
"Kündigungsfristen" or "Probezeit", which the paragraph text itself often lacks.

In [ ]:
# to_documents comes from lexgraph/rag/ingest.py (imported in section 1): one Document per article,
# indexed text = "Art. <number> OR <heading path>" plus the article text, metadata = id, law, article, area, title, url.
docs = to_documents(articles)
print(len(docs), "chunks")
print(docs[0].page_content[:300])

## 4. Dense retrieval with embeddings

Embeddings find articles that mean the same thing even when the words differ ("entlassen" vs. "kündigen").
The Apertus Swiss embedding model runs on the FHNW proxy. With `EMBEDDING_BACKEND = "local"` a multilingual
sentence-transformers model runs inside Deepnote instead.

In [ ]:
from langchain_community.vectorstores import FAISS

if EMBEDDING_BACKEND == "proxy":
    from langchain_openai import OpenAIEmbeddings
    embeddings = OpenAIEmbeddings(
        model=EMBED_MODEL,
        base_url=LLM_BASE_URL,
        api_key=os.environ["LITELLM_API_KEY"],
        check_embedding_ctx_length=False,   # needed for non-OpenAI models
    )
else:
    from langchain_huggingface import HuggingFaceEmbeddings
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

vectorstore = FAISS.from_documents(docs, embeddings)
for hit in vectorstore.similarity_search("Darf mein Chef mir kündigen, wenn ich krank bin?", k=3):
    print(hit.metadata["id"], "|", hit.metadata["title"])

## 5. Keyword retrieval with BM25 and hybrid fusion

Lawyers search with exact terms and article numbers. Embeddings are weak at that, BM25 is strong at it.
We combine both rankings with Reciprocal Rank Fusion: each list gives `1 / (60 + rank)` points per article.

In [ ]:
from rank_bm25 import BM25Okapi

def tokenize(text):
    return re.findall(r"\w+", text.lower())

bm25 = BM25Okapi([tokenize(d.page_content) for d in docs])
doc_by_id = {d.metadata["id"]: d for d in docs}
doc_ids = [d.metadata["id"] for d in docs]

def bm25_ids(query, k=10):
    scores = bm25.get_scores(tokenize(query))
    ranked = sorted(range(len(docs)), key=lambda i: scores[i], reverse=True)
    return [doc_ids[i] for i in ranked[:k] if scores[i] > 0]

def rrf(rankings, k=60):
    """Reciprocal Rank Fusion of several ranked id lists."""
    scores = {}
    for ranking in rankings:
        for rank, doc_id in enumerate(ranking, start=1):
            scores[doc_id] = scores.get(doc_id, 0) + 1 / (k + rank)
    return sorted(scores, key=scores.get, reverse=True)

print(bm25_ids("Kündigungsfrist Wohnung"))
print(rrf([["A", "B", "C"], ["C", "A"]]))

In [ ]:
def dense_ids(query, k=10):
    return [d.metadata["id"] for d in vectorstore.similarity_search(query, k=k)]

def retrieve(query, k=4, mode="hybrid"):
    """Return the top-k articles as Documents. mode: 'dense', 'bm25' or 'hybrid'."""
    if mode == "dense":
        ids = dense_ids(query)
    elif mode == "bm25":
        ids = bm25_ids(query)
    else:
        ids = rrf([dense_ids(query), bm25_ids(query)])
    hits = [doc_by_id[i] for i in ids]
    if USE_RERANKER:
        hits = rerank(query, hits)
    return hits[:k]

for d in retrieve("Mieter zahlt die Miete nicht, was kann der Vermieter tun?"):
    print(d.metadata["id"], "|", d.metadata["title"])

## 6. Reranking with a cross-encoder (optional)

A bi-encoder embeds question and article separately, which is fast but coarse. A cross-encoder reads question and
article together and scores the pair, which is slower but more precise (lecture 13). We only rerank the top
candidates. Switch on with `USE_RERANKER = True`.

In [ ]:
if USE_RERANKER:
    from sentence_transformers import CrossEncoder
    cross_encoder = CrossEncoder("cross-encoder/mmarco-mMiniLMv2-L12-H384-v1")

def rerank(query, hits):
    scores = cross_encoder.predict([(query, d.page_content) for d in hits])
    return [d for _, d in sorted(zip(scores, hits), key=lambda pair: pair[0], reverse=True)]

## 7. Measuring retrieval: Hit@k

Hit@k is the share of questions for which the correct article is among the top k results. It is cheap and does
not need an LLM, so we measure retrieval separately from answer quality. The final test set lives in notebook 06.

**Open for T12.** The six `RETRIEVAL_CASES` below still come from the paraphrase phase. T12 switches the measurement to
the ten cases in `data/testset.jsonl` (`question`, `gold_articles`). Baseline on the real Fedlex text: BM25 Hit@3 = 5/10,
because lay words such as "Schimmel" or "Kaution" do not occur in the law.

In [ ]:
RETRIEVAL_CASES = [
    ("Wie lange ist die Kündigungsfrist im dritten Dienstjahr?", "OR-335c"),
    ("Darf der Arbeitgeber kündigen, während ich krank bin?", "OR-336c"),
    ("Fristlose Entlassung nach Diebstahl im Betrieb", "OR-337"),
    ("Mieter zahlt die Miete nicht, was kann der Vermieter tun?", "OR-257d"),
    ("Schimmel in der Wohnung, darf ich weniger Miete zahlen?", "OR-259a"),
    ("Mit welcher Frist kann ich meine Wohnung kündigen?", "OR-266c"),
]

def hit_at_k(ranked_ids, gold_id, k):
    return gold_id in ranked_ids[:k]

def evaluate_retrieval(search, k=3):
    hits = [hit_at_k(search(q), gold, k) for q, gold in RETRIEVAL_CASES]
    return sum(hits) / len(hits)

print("BM25 Hit@3:", round(evaluate_retrieval(bm25_ids), 2))

In [ ]:
import pandas as pd

pd.DataFrame([
    {"mode": mode, "Hit@1": evaluate_retrieval(lambda q: [d.metadata["id"] for d in retrieve(q, 10, mode)], 1),
     "Hit@3": evaluate_retrieval(lambda q: [d.metadata["id"] for d in retrieve(q, 10, mode)], 3)}
    for mode in ["bm25", "dense", "hybrid"]
])

## 8. Save the index

Notebooks 03 and 04 and the MCP server load the saved index instead of rebuilding it.

In [ ]:
vectorstore.save_local("index/faiss_or")
print("saved to index/faiss_or")

## Your turn

- Find a question where BM25 wins and one where the embeddings win. What do they have in common?
- Which chunking would you choose for long articles such as Art. 362 OR (a list of articles), and why?
- On the real Fedlex text BM25 reaches only Hit@3 = 5/10 (T09 test set). Which lay words of the questions are missing
  in the law, and which retrieval step from sections 4 to 6 should close that gap?

## Decision log

| Date | Decision | Rejected alternative | Reason |
|---|---|---|---|
| Oct 2026 | One chunk per article | Fixed 500-token chunks | The article is the citation unit. Fixed chunks cut articles and break the citation check. |
| 30 Oct 2026 | T11: knowledge base from the Fedlex Akoma Ntoso XML of the OR, pinned to the consolidation of 1 Oct 2026 | Scrape the Fedlex web page (HTML), or always fetch the latest version | The XML has stable article ids (`art_335_c`) and separates footnotes from the text; the web page is rendered by JavaScript. A pinned version keeps index, test set and evaluation on the same text. Standard library only, no new package. |
| 30 Oct 2026 | T11: one chunk per article with the heading path (`context`) in the indexed text; repealed articles left out | Fixed-length chunks, or one chunk per paragraph | A hit maps directly to the id the citation check compares. 208 of 216 articles are under 1500 characters. The heading path adds words the paragraphs lack. Per-paragraph splitting stays open for T12. |
| 30 Oct 2026 | Notebook 02 reads the knowledge base from the repository (`lexgraph.rag.ingest`) instead of defining its own sample records | Keep the sample records in the notebook | One source for notebook, MCP server and tests. The paraphrases stay only as offline test data in the repository. |
| | Embedding model: | | |
| | Hybrid retrieval: yes / no | | Hit@k from section 7 |
| | Reranking: yes / no | | Hit@k gain vs. latency |

## Building block to take away

`retrieve(query, k, mode)` from section 5. It moves into the repository as `rag/retrieval.py` and is wrapped as an
MCP tool in notebook 03.

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=9b9d17b7-b4fd-473b-beb2-f7ed180fbdd3' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>